# Bird Audio Batch Analysis - Gaulossen NTNU Project

**Recordings:** May 13-15, Gaulossen, Norway  
**Purpose:** Batch processing of bird audio recordings with BirdNET analysis

This notebook processes multiple audio files individually (no merging), analyzes them with BirdNET, and produces:
- Individual file analysis with proper naming
- Audacity labels for each file
- Detection visualizations (spectrograms, waveforms)
- F0 and formant analysis
- Summary CSV with all detections

---

## Workflow

1. **Setup & Configuration** - Set paths and analysis parameters
2. **BirdNET Analysis** - Run BirdNET on each file individually
3. **Visualize Detections** - Generate spectrograms and waveforms with F0/formant analysis
4. **Overview Reports** - Create summary CSV and overview visualizations

---

**License:** MIT License  
**Credit:** George Redpath (2025)

## Cell 1: Setup & Configuration

In [ ]:
import os
import glob
import pandas as pd
from pathlib import Path

# ============================================================
# USER CONFIGURATION
# ============================================================

# Directory containing your audio files
AUDIO_DIR = "../audio_files"  # Place your WAV/MP3 files here

# Output directory for all results
RESULTS_DIR = "../results"

# File pattern to match (e.g., "*.wav", "*.mp3", or "*" for all)
FILE_PATTERN = "*.wav"

# BirdNET Settings
MIN_CONF = 0.25  # Minimum confidence threshold (0.0-1.0)

# Location context (helps BirdNET filter to likely species)
# Gaulossen, Trondheim area coordinates:
LAT = 63.4305  # Gaulossen latitude
LON = 10.3951  # Gaulossen longitude
DATE = "2025-05-15"  # Recording date (YYYY-MM-DD)

# Optional: Filter to specific species (None = all species)
SPECIES_FILTER = None  # e.g., ["Common Blackbird", "Great Tit"]

# F0 Analysis Range (adjust for expected bird species pitch range)
F0_MIN_HZ = 500.0
F0_MAX_HZ = 8000.0

# Visualization Settings
PAD_BEFORE_S = 0.3  # Seconds of context before detection
PAD_AFTER_S = 0.3   # Seconds of context after detection
MAX_DETECTIONS_PER_FILE = None  # None = all detections

# ============================================================
# Setup
# ============================================================

# Create output directories
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "labels"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "visualizations"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "csvs"), exist_ok=True)

# Find all audio files
audio_files = sorted(glob.glob(os.path.join(AUDIO_DIR, FILE_PATTERN)))

print(f"Found {len(audio_files)} audio files to process:")
for i, f in enumerate(audio_files, 1):
    print(f"  {i}. {os.path.basename(f)}")

if len(audio_files) == 0:
    print(f"\n⚠️  No files found in {AUDIO_DIR} matching pattern '{FILE_PATTERN}'")
    print(f"   Please place your audio files in the audio_files/ directory")

## Cell 2: BirdNET Analysis (Batch Processing)

In [ ]:
from birdnetlib import Recording
from birdnetlib.analyzer import Analyzer
from datetime import datetime
import json

# ============================================================
# Initialize BirdNET
# ============================================================

print("Initializing BirdNET analyzer...")
analyzer = Analyzer()

# Parse date if provided
analysis_date = None
if DATE:
    try:
        analysis_date = datetime.strptime(DATE, "%Y-%m-%d")
    except:
        print(f"⚠️  Could not parse date '{DATE}', continuing without date context")

# ============================================================
# Process each audio file
# ============================================================

all_detections = []
file_summaries = []

for file_idx, audio_path in enumerate(audio_files, 1):
    filename = os.path.basename(audio_path)
    file_stem = Path(audio_path).stem
    
    print(f"\n[{file_idx}/{len(audio_files)}] Analyzing: {filename}")
    
    try:
        # Run BirdNET analysis
        recording = Recording(
            analyzer,
            audio_path,
            lat=LAT,
            lon=LON,
            date=analysis_date,
            min_conf=MIN_CONF,
            species_list=SPECIES_FILTER
        )
        recording.analyze()
        
        # Extract detections
        file_detections = []
        for detection in recording.detections:
            det_data = {
                "file_index": file_idx,
                "filename": filename,
                "file_stem": file_stem,
                "start_s": float(detection["start_time"]),
                "end_s": float(detection["end_time"]),
                "common_name": detection.get("common_name", ""),
                "scientific_name": detection.get("scientific_name", ""),
                "confidence": float(detection.get("confidence", 0.0)),
                "label": detection.get("label", "")
            }
            file_detections.append(det_data)
            all_detections.append(det_data)
        
        # Create Audacity label file for this audio file
        label_path = os.path.join(RESULTS_DIR, "labels", f"{file_stem}_labels.txt")
        with open(label_path, "w", encoding="utf-8") as f:
            for det in file_detections:
                label_text = f"{det['common_name']} ({det['confidence']:.2f})"
                f.write(f"{det['start_s']:.6f}\t{det['end_s']:.6f}\t{label_text}\n")
        
        # Save per-file CSV
        if file_detections:
            csv_path = os.path.join(RESULTS_DIR, "csvs", f"{file_stem}_detections.csv")
            pd.DataFrame(file_detections).to_csv(csv_path, index=False)
        
        # File summary
        file_summaries.append({
            "file_index": file_idx,
            "filename": filename,
            "num_detections": len(file_detections),
            "unique_species": len(set(d["common_name"] for d in file_detections)),
            "label_file": label_path,
        })
        
        print(f"   Found {len(file_detections)} detections")
        if file_detections:
            species_counts = {}
            for d in file_detections:
                sp = d["common_name"]
                species_counts[sp] = species_counts.get(sp, 0) + 1
            for sp, count in sorted(species_counts.items()):
                print(f"     - {sp}: {count}")
        
    except Exception as e:
        print(f"   ⚠️  Error processing {filename}: {e}")
        file_summaries.append({
            "file_index": file_idx,
            "filename": filename,
            "num_detections": 0,
            "unique_species": 0,
            "error": str(e)
        })

# ============================================================
# Save master detection CSV
# ============================================================

master_csv = os.path.join(RESULTS_DIR, "all_detections.csv")
if all_detections:
    df_all = pd.DataFrame(all_detections)
    df_all = df_all.sort_values(["file_index", "start_s"]).reset_index(drop=True)
    df_all.to_csv(master_csv, index=False)
    print(f"\n✅ Saved master detections CSV: {master_csv}")
    print(f"   Total detections: {len(all_detections)}")
else:
    print("\nℹ️  No detections found across all files")

# Save file summary
summary_csv = os.path.join(RESULTS_DIR, "file_summary.csv")
pd.DataFrame(file_summaries).to_csv(summary_csv, index=False)
print(f"✅ Saved file summary: {summary_csv}")

## Cell 2b: Export to Raven Pro Selection Tables (Optional)

In [ ]:
# ============================================================
# Export BirdNET Detections to Raven Pro Selection Tables
# ============================================================
# This cell exports your BirdNET detections to Raven Pro format.
# Raven Pro is professional bioacoustic analysis software from Cornell Lab.
# You can skip this cell if you only need Audacity labels.

if not all_detections:
    print("No detections to export to Raven format")
else:
    # Create Raven output directory
    raven_dir = os.path.join(RESULTS_DIR, "raven_tables")
    os.makedirs(raven_dir, exist_ok=True)
    
    print(f"Exporting {len(all_detections)} detections to Raven Pro format...\n")
    
    # Raven selection table columns
    RAVEN_COLUMNS = [
        "Selection",
        "View",
        "Channel",
        "Begin Time (s)",
        "End Time (s)",
        "Low Freq (Hz)",
        "High Freq (Hz)",
        "Begin File",
        "Begin Path",
        "File Offset (s)",
        "Common Name",
        "Scientific Name",
        "Confidence"
    ]
    
    # Group detections by file
    df_all_dets = pd.DataFrame(all_detections)
    
    for filename in df_all_dets['filename'].unique():
        file_detections = df_all_dets[df_all_dets['filename'] == filename].copy()
        file_stem = Path(filename).stem
        audio_path = os.path.abspath(os.path.join(AUDIO_DIR, filename))
        
        # Build Raven selection table
        raven_rows = []
        for idx, det in file_detections.iterrows():
            # Estimate frequency bounds from F0 if available
            # Default to bird vocalization range if not computed
            low_freq = F0_MIN_HZ
            high_freq = F0_MAX_HZ
            
            raven_row = {
                "Selection": idx + 1,
                "View": "Spectrogram 1",
                "Channel": 1,
                "Begin Time (s)": det['start_s'],
                "End Time (s)": det['end_s'],
                "Low Freq (Hz)": low_freq,
                "High Freq (Hz)": high_freq,
                "Begin File": filename,
                "Begin Path": audio_path,
                "File Offset (s)": 0.0,
                "Common Name": det.get('common_name', ''),
                "Scientific Name": det.get('scientific_name', ''),
                "Confidence": det.get('confidence', 0.0)
            }
            raven_rows.append(raven_row)
        
        # Create DataFrame and save as tab-delimited text
        df_raven = pd.DataFrame(raven_rows, columns=RAVEN_COLUMNS)
        raven_path = os.path.join(raven_dir, f"{file_stem}_raven.txt")
        df_raven.to_csv(raven_path, sep="\t", index=False)
        
        print(f"✅ {filename}: {len(raven_rows)} selections -> {raven_path}")
    
    print(f"\n✅ Raven selection tables saved to: {raven_dir}")
    print("\n💡 To use in Raven Pro:")
    print("   1. Open Raven Pro")
    print("   2. File → Open Selection Table")
    print("   3. Select the *_raven.txt file")
    print("   4. Raven will load the audio file and display all selections")

## Cell 3: Visualize Detections (Spectrograms + F0/Formant Analysis)

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from scipy.signal import lfilter, hamming

# ============================================================
# Visualization Helper Functions
# ============================================================

def estimate_f0(y_seg, sr, fmin=500, fmax=8000):
    """Estimate F0 using librosa.pyin"""
    try:
        f0, voiced_flag, voiced_prob = librosa.pyin(
            y_seg,
            fmin=fmin,
            fmax=fmax,
            frame_length=2048,
            hop_length=512
        )
        return f0
    except Exception as e:
        return None

def estimate_formants(y_seg, sr, order=12, preemph=0.97):
    """Estimate formants F1, F2, F3 using LPC"""
    win_length = 2048
    
    if len(y_seg) < win_length:
        x = y_seg.copy()
    else:
        center = len(y_seg) // 2
        half = win_length // 2
        x = y_seg[max(0, center - half): center + half]
    
    if len(x) < 32:
        return [np.nan, np.nan, np.nan]
    
    # Pre-emphasis and windowing
    x = lfilter([1, -preemph], 1, x)
    x = x * hamming(len(x), sym=False)
    
    try:
        a = librosa.lpc(x, order=order)
        roots = np.roots(a)
        roots = [r for r in roots if np.imag(r) >= 0.01]
        angs = np.arctan2(np.imag(roots), np.real(roots))
        freqs = np.sort(angs * (sr / (2 * np.pi)))
        freqs = freqs[(freqs > 0) & (freqs < sr / 2.0)]
        
        out = list(freqs[:3])
        while len(out) < 3:
            out.append(np.nan)
        return out
    except:
        return [np.nan, np.nan, np.nan]

def plot_detection(audio_path, start_s, end_s, species, conf, out_path, sr_target=44100):
    """Create waveform and spectrogram for a detection"""
    # Load audio segment
    y, sr = librosa.load(audio_path, sr=sr_target, mono=True)
    
    # Extract segment with padding
    start_idx = int(max(0, (start_s - PAD_BEFORE_S) * sr))
    end_idx = int(min(len(y), (end_s + PAD_AFTER_S) * sr))
    y_seg = y[start_idx:end_idx]
    
    if len(y_seg) < 100:
        return None
    
    # F0 estimation
    f0_track = estimate_f0(y_seg, sr, fmin=F0_MIN_HZ, fmax=F0_MAX_HZ)
    f0_mean = np.nan
    if f0_track is not None:
        f0_vals = f0_track[~np.isnan(f0_track)]
        if len(f0_vals) > 0:
            f0_mean = float(np.mean(f0_vals))
    
    # Formant estimation
    F1, F2, F3 = estimate_formants(y_seg, sr)
    
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
    # Waveform
    t = np.arange(len(y_seg)) / sr
    ax1.plot(t, y_seg)
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Amplitude")
    ax1.set_title(f"{species} (conf {conf:.2f}) - Waveform")
    ax1.grid(True, alpha=0.3)
    
    # Spectrogram
    S = np.abs(librosa.stft(y_seg, n_fft=2048, hop_length=512, win_length=2048))
    S_db = librosa.amplitude_to_db(S, ref=np.max)
    
    img = librosa.display.specshow(S_db, sr=sr, hop_length=512, x_axis="time", y_axis="hz", ax=ax2)
    fig.colorbar(img, ax=ax2, format="%+2.0f dB")
    
    # Overlay F0 track if available
    if f0_track is not None:
        times = librosa.times_like(f0_track, sr=sr, hop_length=512)
        mask = ~np.isnan(f0_track)
        if np.any(mask):
            ax2.plot(times[mask], f0_track[mask], 'r-', linewidth=2, label='F0', alpha=0.7)
    
    # Add annotation with metrics
    metrics_text = f"F0: {f0_mean:.0f} Hz\nF1: {F1:.0f} Hz\nF2: {F2:.0f} Hz\nF3: {F3:.0f} Hz"
    ax2.text(0.02, 0.98, metrics_text, transform=ax2.transAxes,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
             fontsize=9)
    
    ax2.set_title(f"{species} (conf {conf:.2f}) - Spectrogram with F0")
    ax2.legend(loc='upper right')
    
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    return {"f0_mean": f0_mean, "F1": F1, "F2": F2, "F3": F3}

# ============================================================
# Generate visualizations for all detections
# ============================================================

if not all_detections:
    print("No detections to visualize")
else:
    print(f"Generating visualizations for {len(all_detections)} detections...\n")
    
    visualization_data = []
    
    for idx, det in enumerate(all_detections):
        # Limit per file if specified
        if MAX_DETECTIONS_PER_FILE is not None:
            file_dets = [d for d in all_detections if d["filename"] == det["filename"]]
            if file_dets.index(det) >= MAX_DETECTIONS_PER_FILE:
                continue
        
        audio_path = os.path.join(AUDIO_DIR, det["filename"])
        out_filename = f"{det['file_stem']}_det{idx:04d}_{det['common_name'].replace(' ', '_')}.png"
        out_path = os.path.join(RESULTS_DIR, "visualizations", out_filename)
        
        print(f"[{idx + 1}/{len(all_detections)}] {det['filename']} - {det['common_name']}")
        
        try:
            metrics = plot_detection(
                audio_path,
                det["start_s"],
                det["end_s"],
                det["common_name"],
                det["confidence"],
                out_path
            )
            
            if metrics:
                visualization_data.append({
                    **det,
                    **metrics,
                    "visualization_path": out_path
                })
        except Exception as e:
            print(f"   ⚠️  Error: {e}")
    
    # Save visualization summary
    if visualization_data:
        vis_csv = os.path.join(RESULTS_DIR, "detections_with_metrics.csv")
        pd.DataFrame(visualization_data).to_csv(vis_csv, index=False)
        print(f"\n✅ Saved visualization metrics: {vis_csv}")
    
    print(f"\n✅ Visualizations saved to: {os.path.join(RESULTS_DIR, 'visualizations')}")

## Cell 4: Summary Statistics & Species Overview

In [ ]:
import matplotlib.pyplot as plt

if not all_detections:
    print("No detections to summarize")
else:
    df = pd.DataFrame(all_detections)
    
    print("="*60)
    print("GAULOSSEN BIRD SURVEY SUMMARY")
    print("="*60)
    print(f"\nRecording Period: May 13-15, 2025")
    print(f"Location: Gaulossen, Trondheim (63.4305°N, 10.3951°E)")
    print(f"\nFiles Processed: {len(audio_files)}")
    print(f"Total Detections: {len(all_detections)}")
    print(f"Unique Species: {df['common_name'].nunique()}")
    print(f"Confidence Threshold: {MIN_CONF}")
    
    print("\n" + "="*60)
    print("SPECIES COUNTS")
    print("="*60)
    species_counts = df['common_name'].value_counts()
    for species, count in species_counts.items():
        avg_conf = df[df['common_name'] == species]['confidence'].mean()
        print(f"{species:30s} {count:4d} detections (avg conf: {avg_conf:.2f})")
    
    print("\n" + "="*60)
    print("PER-FILE BREAKDOWN")
    print("="*60)
    for filename in df['filename'].unique():
        file_df = df[df['filename'] == filename]
        print(f"\n{filename}:")
        print(f"  Total detections: {len(file_df)}")
        print(f"  Unique species: {file_df['common_name'].nunique()}")
        file_species = file_df['common_name'].value_counts()
        for sp, cnt in file_species.head(5).items():
            print(f"    - {sp}: {cnt}")
    
    # Create species bar chart
    plt.figure(figsize=(14, 6))
    species_counts.head(20).plot(kind='barh')
    plt.xlabel('Number of Detections')
    plt.title(f'Top 20 Bird Species Detected - Gaulossen (May 13-15, 2025)\nTotal: {len(all_detections)} detections across {len(audio_files)} files')
    plt.tight_layout()
    
    chart_path = os.path.join(RESULTS_DIR, "species_summary.png")
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n✅ Species summary chart saved: {chart_path}")
    
    print("\n" + "="*60)
    print("OUTPUT FILES")
    print("="*60)
    print(f"Master CSV: {os.path.join(RESULTS_DIR, 'all_detections.csv')}")
    print(f"File Summary: {os.path.join(RESULTS_DIR, 'file_summary.csv')}")
    print(f"Audacity Labels: {os.path.join(RESULTS_DIR, 'labels')}/ (one per audio file)")
    print(f"Visualizations: {os.path.join(RESULTS_DIR, 'visualizations')}/")
    print(f"Per-File CSVs: {os.path.join(RESULTS_DIR, 'csvs')}/")

---

## MIT License

Copyright (c) 2025 George Redpath

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

---

**Credit:** Please credit George Redpath when using this code.